# 33 — Earth analog review for rover route-planning practice

**Question:** which Earth terrain could reproduce the broad route-planning challenge of a lunar patch? Use the [mission exercise and relevance rubric](../../../outline.md#first-similarity-use-case-rover-route-planning-practice). This notebook prepares visual evidence and blank review forms, not a new training experiment or a validated ranking score.

The review concerns 960 m terrain patches on a 30 m grid. Review physical scale, landform arrangement and broad route-planning choices separately. Unknown wheel-scale obstacles, traction and equipment limits prevent passability claims. Similarity is not a safety or site-access rating.

## Fixed selection before viewing results

Use the existing notebook-29 query sample. Select the lunar holdout region plus the lowest- and highest-median-relief training regions in the full-context query sample. Within each selected region and each lunar view, choose the query nearest that region/view's median relief, with patch-ID ties. Views can select different patches; they are separate review cases, not paired causal comparisons.

For each query, pool the top Earth candidate from the physical baseline, untrained encoder and both trained sampling variants. Deduplicate candidates and add the best physical-baseline candidate from each Earth region not yet represented. These additions ensure five-site review coverage, not five proven matches. Keep their origin in a separate key. Do not select cases based on favorable CNN outcomes.

**This qualitative review deliberately pools all five existing Earth development regions.** Training and holdout candidates are mixed and recorded in the key; these results are not notebook-29 split metrics or generalization estimates. No final reserves are opened, no weights change, and no new physical compatibility filter is applied.

In [6]:
import json
import sys
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "outline.md").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
from analysis_artifacts import rebase_links
from inspection import check_input, file_hash, verify_run
from similarity_cnn import cosine_top_k
from terrain_dataset import FEATURES, validate_config
from terrain_learning import rank_candidates

SOURCE = ROOT / "data/processed/terrain_similarity_cnn/20261101T082707986851Z_5413e6"
DATASET = ROOT / "data/processed/terrain_learning_readiness/20261101T064233918547Z_4a05ac"
NOTEBOOK = ROOT / "notebooks/modeling/terrain_similarity/33_mission_analog_review.ipynb"
OUT = ROOT / "data/processed/analysis" / ("mission-analog-review-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ"))
OUT.mkdir(parents=True, exist_ok=False)
expected = {**verify_run(SOURCE), **verify_run(DATASET)}
evidence = {str(p): file_hash(p) for p in [SOURCE / "provenance.json", DATASET / "provenance.json", NOTEBOOK, ROOT / "outline.md",
    *(ROOT / "notebooks" / n for n in ["inspection.py", "similarity_cnn.py", "terrain_dataset.py", "terrain_learning.py", "retrieval_analysis.py"])]}
config = json.loads((DATASET / "provenance.json").read_text(encoding="utf-8"))["config"]
validate_config(config)
assert config["patch_size_m"] == 960 and config["spacing_m"] == 30
patches = pd.read_csv(check_input(DATASET / "patches.csv", expected, evidence)).set_index("patch_id", verify_integrity=True)
roles = {(r["body"], r["region"]): r["fold"] for r in config["regions"]}
for item in patches[["body", "region", "role", "fold"]].drop_duplicates().to_dict("records"):
    assert item["role"] == "development" and roles.get((item["body"], item["region"])) == item["fold"]
full_queries = pd.read_csv(check_input(SOURCE / "full_context/evaluation_queries.csv", expected, evidence))
selection = patches.loc[full_queries.loc[full_queries["body"] == "moon", "patch_id"]]
training = selection.loc[selection["fold"] == "train"].groupby("region")["central_relief_m"].median().sort_values(kind="stable")
holdout_regions = selection.loc[selection["fold"] == "holdout", "region"].unique()
assert len(holdout_regions) == 1
regions = [holdout_regions[0], training.index[0], training.index[-1]]
assert len(set(regions)) == 3
rng = np.random.default_rng(33)
provenance = {"status": "running", "source": str(SOURCE.relative_to(ROOT)), "dataset": str(DATASET.relative_to(ROOT)),
    "regions": regions, "candidate_pool": "all five Earth development regions; train/holdout mixed for qualitative review only",
    "method_blinding": "random candidate order seed 33; methods and geography in separate key; preparer is not blind",
    "independent_labels": False, "reserves_opened": False, "python": sys.version, "numpy": np.__version__, "pandas": pd.__version__}
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")

742

In [7]:
pair_rows, query_rows = [], []
for view in ["full_context", "image_supported"]:
    query_file = pd.read_csv(check_input(SOURCE / view / "evaluation_queries.csv", expected, evidence))
    queries = patches.loc[query_file.loc[query_file["body"] == "moon", "patch_id"]]
    banks = {}
    for sampling in ["region_balanced", "count_weighted"]:
        with np.load(check_input(SOURCE / view / sampling / "embeddings.npz", expected, evidence), allow_pickle=False) as archive:
            banks[sampling] = {name: archive[name].copy() for name in ["patch_ids", "cnn", "random_encoder"]}
    ids = banks["region_balanced"]["patch_ids"]
    assert len(set(ids)) == len(ids)
    np.testing.assert_array_equal(ids, banks["count_weighted"]["patch_ids"])
    np.testing.assert_allclose(banks["region_balanced"]["random_encoder"], banks["count_weighted"]["random_encoder"])
    bank = patches.loc[ids]
    assert set(ids) == set(patches.index[patches["view"].isin(["natural_screen", view])])
    pool = np.flatnonzero(bank["body"].to_numpy() == "earth")
    earth = bank.iloc[pool]
    assert earth["region"].nunique() == 5
    candidate_ids = earth.index.tolist()
    descriptors = bank[FEATURES].to_numpy(dtype=float)
    for region in regions:
        region_queries = queries.loc[queries["region"] == region].copy()
        target = region_queries["central_relief_m"].median()
        region_queries["median_distance"] = abs(region_queries["central_relief_m"] - target)
        query_id = region_queries.reset_index().sort_values(["median_distance", "patch_id"]).iloc[0]["patch_id"]
        qi = bank.index.get_loc(query_id)
        assert isinstance(qi, (int, np.integer))
        physical_order, _ = rank_candidates(descriptors[qi], descriptors[pool], candidate_ids, len(pool))
        origins = {candidate_ids[physical_order[0]]: ["physical_top1"]}
        for sampling in ["region_balanced", "count_weighted"]:
            order, _ = cosine_top_k(banks[sampling]["cnn"][qi], banks[sampling]["cnn"][pool], candidate_ids)
            origins.setdefault(candidate_ids[order[0]], []).append("cnn_" + sampling + "_top1")
        order, _ = cosine_top_k(banks["region_balanced"]["random_encoder"][qi], banks["region_balanced"]["random_encoder"][pool], candidate_ids)
        origins.setdefault(candidate_ids[order[0]], []).append("random_encoder_top1")
        represented = set(patches.loc[list(origins), "region"])
        for earth_region in sorted(set(earth["region"]) - represented):
            index = next(i for i in physical_order if earth.iloc[i]["region"] == earth_region)
            origins[candidate_ids[index]] = ["geographic_supplement_physical_best_in_region"]
        assert patches.loc[list(origins), "region"].nunique() == 5
        group_id = f"Q{len(query_rows) + 1:02d}"
        query_rows.append({"group_id": group_id, "query_id": query_id, "query_region": region, "query_fold": bank.iloc[qi]["fold"],
                           "view": view, "candidate_count": len(origins), "gallery": group_id + ".png"})
        shuffled = rng.permutation(sorted(origins))
        for index, candidate_id in enumerate(shuffled, 1):
            pair_rows.append({"pair_id": f"{group_id}C{index:02d}", "group_id": group_id, "query_id": query_id,
                "candidate_id": candidate_id, "candidate_region": patches.loc[candidate_id, "region"],
                "candidate_fold": patches.loc[candidate_id, "fold"], "origins": "|".join(origins[candidate_id]), "view": view})
pairs = pd.DataFrame(pair_rows)
query_groups = pd.DataFrame(query_rows)
assert len(query_groups) == 6 and pairs["pair_id"].is_unique
assert not pairs.duplicated(["query_id", "candidate_id"]).any()
pairs.to_csv(OUT / "method_key.csv", index=False)
query_groups.to_csv(OUT / "query_key.csv", index=False)
ratings = pairs[["pair_id", "group_id"]].copy()
for field in ["physical_scale_0_1_2_U", "landform_arrangement_0_1_2_U", "route_challenge_0_1_2_U", "evidence_confidence", "reason", "reviewer", "review_date"]:
    ratings[field] = ""
ratings.to_csv(OUT / "pair_review_blank.csv", index=False)
decisions = query_groups[["group_id", "gallery"]].copy()
for field in ["promising_partial_none_uncertain", "preferred_pair_ids", "missing_condition", "reviewer", "review_date"]:
    decisions[field] = ""
decisions.to_csv(OUT / "query_review_blank.csv", index=False)
display(query_groups[["group_id", "query_region", "query_fold", "view", "candidate_count"]])

,group_id,query_region,query_fold,view,candidate_count
0,Q01,A3CLR22_10_Leibnitz_Beta_Plateau,holdout,full_context,8
1,Q02,A3CLR22_2_Haworth,train,full_context,7
2,Q03,A3CLR22_11_Malapert_Massif,train,full_context,7
3,Q04,A3CLR22_10_Leibnitz_Beta_Plateau,holdout,image_supported,8
4,Q05,A3CLR22_2_Haworth,train,image_supported,6
5,Q06,A3CLR22_11_Malapert_Massif,train,image_supported,8


## Review instructions

Open the gallery cards and the rubric before `method_key.csv` or `query_key.csv`. Candidate order is randomized and method/rank/site names are concealed in the cards. Support view is visible because it affects evidence confidence. This reduces method cues for a new reviewer; the preparer has seen the method key and is not an independent blinded reviewer.

Each column is a patch with the same 960 m ground extent, in its original raster orientation (grid-up), without geographic-north alignment. Relative elevation uses a shared symmetric colour range within each card; slope shares a 0-to-maximum range. No per-patch height stretching or rotation is applied. Read the labelled ranges before comparing colours across cards. Numbers are measurements, not scores. Local SD is recorded numerically; it does not establish rocks or surface traction.

Copy the blank forms before entering judgments. Complete pair dimensions with 0/1/2/U and a reason; then give a query-level decision including “none among reviewed candidates” or “uncertain” when appropriate. Leave unsupported dimensions U. Ideally two reviewers assess the same cases independently before discussing disagreements. Only afterwards reveal methods and geographic supplements. These supplements must not be counted as retrieved top-one successes. Do not manufacture no-match labels from descriptor tolerances.

In [8]:
cache = {}
measurements = []
for query in query_groups.to_dict("records"):
    group = pairs.loc[pairs["group_id"] == query["group_id"]]
    plot_ids = [query["query_id"], *group["candidate_id"].tolist()]
    labels = ["Moon query " + query["group_id"], *group["pair_id"].tolist()]
    arrays = []
    for patch_id in plot_ids:
        item = patches.loc[patch_id]
        if item["shard"] not in cache:
            with np.load(check_input(DATASET / item["shard"], expected, evidence), allow_pickle=False) as archive:
                assert archive["valid"].all()
                cache[item["shard"]] = archive["physical"].copy()
        arrays.append(cache[item["shard"]][int(item["tensor_index"])])
    assert all(a.shape == (3, 32, 32) and np.isfinite(a).all() for a in arrays)
    limit = max(float(np.abs(a[0]).max()) for a in arrays) or 1.
    slope_limit = max(float(a[1].max()) for a in arrays) or 1.
    fig, axes = plt.subplots(2, len(arrays), figsize=(3.0 * len(arrays), 7), constrained_layout=True)
    for column, (patch_id, label, values) in enumerate(zip(plot_ids, labels, arrays, strict=True)):
        record = patches.loc[patch_id]
        height_image = axes[0, column].imshow(values[0], cmap="RdBu_r", vmin=-limit, vmax=limit, extent=(0, 960, 0, 960))
        slope_image = axes[1, column].imshow(values[1], cmap="viridis", vmin=0, vmax=slope_limit, extent=(0, 960, 0, 960))
        axes[0, column].set_title(f"{label}\nRelief {record['central_relief_m']:.0f} m", fontsize=10)
        axes[1, column].set_title(f"P90 slope {record['slope_p90_deg']:.1f} deg\nMedian local SD {record['local_sd_median_m']:.1f} m", fontsize=9)
        for ax in axes[:, column]:
            ax.set(xticks=[0, 960], yticks=[0, 960], xlabel="metres")
        measurements.append({"group_id": query["group_id"], "display_id": label, "central_relief_m": record["central_relief_m"],
                             "slope_p90_deg": record["slope_p90_deg"], "local_sd_median_m": record["local_sd_median_m"]})
    fig.colorbar(height_image, ax=axes[0, :].tolist(), label="Relative elevation (m)", shrink=.8)
    fig.colorbar(slope_image, ax=axes[1, :].tolist(), label="Slope (degrees)", shrink=.8)
    fig.suptitle(query["group_id"] + " | " + query["view"] + " | coarse route-planning morphology; methods concealed", fontsize=13)
    fig.savefig(OUT / query["gallery"], dpi=110)
    plt.close(fig)
pd.DataFrame(measurements).to_csv(OUT / "physical_measurements.csv", index=False)

## What this review can decide

Record observable feature similarities and mismatches before proposing a model change. A strong physical match with a different spatial arrangement may be a poor route-planning analog; an attractive shape at incompatible relief may also be poor. Distinguish both from uncertain source evidence.

This small, deliberately varied set is for refining the rubric and identifying failure modes. It cannot estimate model accuracy or establish a validated winner. Independent judgments are still required. Keep agent observations separate from blank human-review forms, and do not train on them as ground truth.

Next decisions should follow the review: physical filtering if physical mismatch dominates useful shape suggestions; objective/data changes if spatial arrangement is consistently wrong; more development candidates if none reproduce the exercise. Keep environment, access and equipment checks separate from terrain relevance.

# Preparer observations — read after independent review

These are **provisional agent observations**, not independent relevance labels, model accuracy or passability judgments. The preparer knows how candidates were selected. No numeric ratings were written into the review forms. Pair IDs refer to the deterministic notebook-33 selection on the pinned inputs.

| Case | Observable correspondence and remaining concern |
| --- | --- |
| Q01, full-context Leibnitz | Q01C07 is a plausible broad-gradient comparison: relief 147 vs 159 m and P90 slope 14.9 vs 14.8 degrees. Its diagonal slope-band arrangement still needs review. Q01C04 concentrates steepness near an edge, unlike the broad lunar slope. |
| Q02, full-context Haworth | Q02C02 has a broad higher/steeper upper band and lower area, resembling the query at coarse scale. Q02C01 has similar summary magnitudes but a more localized curved steep feature. Slope texture and effective resolution remain uncertain. |
| Q03, full-context Malapert | Q03C03 offers a broad gradient at comparable relief (244 vs 269 m), allowing rotation for this exercise. Its slope field is less uniform. Q03C04 instead concentrates steepness near one corner. |
| Q04, image-supported Leibnitz | Q04C03 and Q04C05 preserve broad high-to-low gradients and substantial relief (308/318 vs 343 m). Earth slope heterogeneity still differs; route equivalence is not established. |
| Q05, image-supported Haworth | Q05C02 is a plausible rotated broad-gradient analog (295 vs 290 m relief), but has more curvature and local slope variation. Q05C01 has much less relief despite a higher P90 slope. |
| Q06, image-supported Malapert | Q06C04 and Q06C07 resemble the broad gradient more closely than Q06C06, which has stronger spatial variation in slope. Neither observation proves equivalent route choices. |

After inspecting the cards, the method key shows Q01C07 came from the region-balanced CNN, Q02C02 from the untrained encoder, and the other highlighted broad-gradient candidates from the physical or untrained controls. This is a small qualitative check, not a tally of wins. Geographic additions also appear in the cards and must not count as model retrieval successes.

**Sampling limitation:** all six selected queries are predominantly broad gradients. This set can expose physical-scale and steep-area-arrangement disagreements, but it does not adequately cover crater, ridge-network or obstacle-rich exercises. Effective-resolution differences may also influence apparent smoothness. Do not infer that smoothness is the learned cause or that filtered CNN retrieval is the solution.

**Next useful evidence:** complete independent rubric judgments on these cards, then select a small additional set for missing landform types from current development regions. Evaluate whether each case actually offers distinct route-planning choices before altering the objective, imposing physical filters or retraining. Equipment passability, access and environmental feasibility remain unresolved.


In [9]:
notebook = nbformat.read(NOTEBOOK, as_version=4)
report = "\n\n".join(c.source for c in notebook.cells if c.cell_type == "markdown" and c.metadata.get("review_stage") != "after")
report = rebase_links(report, NOTEBOOK, OUT / "REPORT.md", {})
report += "\n\n## Review cards\n\n" + "\n\n".join(f"![{r['group_id']}]({r['gallery']})" for r in query_groups.to_dict("records"))
(OUT / "REVIEW.md").write_text(report + "\n", encoding="utf-8")
notes = "\n\n".join(c.source for c in notebook.cells if c.cell_type == "markdown" and c.metadata.get("review_stage") == "after")
notes = rebase_links(notes, NOTEBOOK, OUT / "PREPARER_NOTES.md", {})
(OUT / "PREPARER_NOTES.md").write_text(notes + "\n", encoding="utf-8")
provenance["inputs"] = evidence
provenance["pair_count"] = len(pairs)
provenance["outputs"] = {p.name: file_hash(p) for p in sorted(OUT.iterdir()) if p.is_file() and p.name != "provenance.json"}
provenance["status"] = "publishing"
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print("Complete:", OUT, "pairs:", len(pairs), "independent ratings: none")


Complete: .\data\processed\analysis\mission-analog-review-20261102T143657316675Z pairs: 44 independent ratings: none


## Save reusable data and disposable reports
Run this final cell after the displays. Later notebooks read only the data directory; reports can be deleted and regenerated.

In [10]:
import sys

sys.path.insert(0, str(ROOT / "notebooks"))
from analysis_artifacts import publish_analysis

REPORT_OUT = publish_analysis(ROOT, OUT, manifest_name="provenance.json")
print("Reusable data:", OUT)
print("Disposable reports:", REPORT_OUT)

Reusable data: .\data\processed\analysis\mission-analog-review-20261102T143657316675Z
Disposable reports: .\reports\analysis\mission-analog-review-20261102T143657316675Z
